# Tema 1 — Explorarea corpusului și primul prompt
În acest notebook vei explora corpusul curățat de comentarii YouTube și vei testa un prim prompt exploratoriu.

Vei testa 10 comentarii și vei reflecta asupra unor probleme precum ambiguitatea, țintele multiple, sarcasmul și confuzia dintre sentiment și poziționarea față de țintă.

## 1. Pregătire
Încărcăm bibliotecile necesare și cheia API pentru Gemini.
Modificați doar celula de configurare a studentului.

In [1]:
from pathlib import Path
import os
import json
import random
import pandas as pd
from dotenv import load_dotenv
from openai import OpenAI

In [2]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

load_dotenv(ROOT / ".env")

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "").strip()
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
print("Root project:", ROOT)
print("Gemini key found:", GEMINI_API_KEY is not None)
print("ASCII key:", all(ord(ch) < 128 for ch in GEMINI_API_KEY))

client_gemini = OpenAI(
    api_key=GEMINI_API_KEY,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai"
)

Root project: c:\Users\ioana\Desktop\curs AI Engineering\echochamber-project-team-1
Gemini key found: True
ASCII key: True


## 2. Configurare
Modificați  această celulă.
Schimbați `student_id` cu folderul vostru: `student_01`, `student_02`, etc.

In [3]:
student_id = "student_05"
model = "gemini-2.5-flash-lite"
temperature = 0.2
corpus_file = ROOT / "data" / "cleaned" / "corpus_youtube_large_clean.jsonl"
output_file = ROOT / "outputs" / f"{student_id}_prompt_outputs.jsonl"

## 3. Încărcăm corpusul curățat
Corpusul este salvat în format JSONL.
JSONL înseamnă: un comentariu pe fiecare linie.

In [4]:
# Citim fiecare linie din fișierul JSONL si o transformăm într-un dataframe pentru explorare
records = []
with corpus_file.open("r", encoding="utf-8") as f:
    for line in f:
        records.append(json.loads(line))
# Transformăm lista într-un DataFrame pentru explorare mai ușoară
df = pd.DataFrame(records)
df.head()

,id,source_platform,source_channel,text,text_raw,bubble_label,bubble_self_identified,topic,rhetoric_type,video_id,video_title,video_date,comment_date,likes,lang,collected_at
0,yt_5rHoTX3U_3Q_UgxaV5so7vyeXpyy8up4AaABAg,youtube,georgesimionoficial,Felicitării George Simion Președintele Românie...,Felicitării George Simion Președintele Românie...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-23,1,ro,2026-03-22
1,yt_5rHoTX3U_3Q_UgwJYiRLMbLfl2AipVR4AaABAg,youtube,georgesimionoficial,Asa trebuie să fiți printre oameni nu sa se do...,Asa trebuie să fiți printre oameni nu sa se do...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-02,5,ro,2026-03-22
2,yt_5rHoTX3U_3Q_UgzXqOk_SypZcQS-JcF4AaABAg,youtube,georgesimionoficial,Eu am votat cu George Simion din primul tur pt...,Eu am votat cu George Simion din primul tur pt...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,30,ro,2026-03-22
3,yt_5rHoTX3U_3Q_UgzpKghDX0_l-Gc3P4V4AaABAg,youtube,georgesimionoficial,Si de trebuie deposite de combustibil degeaba ...,Si de trebuie deposite de combustibil degeaba ...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-16,3,ro,2026-03-22
4,yt_5rHoTX3U_3Q_UgwqOTRSHNPj9cuGwNt4AaABAg,youtube,georgesimionoficial,Nu te descuraja că dobitoci și proști vor fi p...,Nu te descuraja că dobitoci și proști vor fi...,None,False,None,None,5rHoTX3U_3Q,Turul României: realități de la firul ierbii,2025-09-30,2025-10-01,9,ro,2026-03-22


In [5]:
print("Number of comments:", len(df))
print("Columns:", list(df.columns))

Number of comments: 30451
Columns: ['id', 'source_platform', 'source_channel', 'text', 'text_raw', 'bubble_label', 'bubble_self_identified', 'topic', 'rhetoric_type', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'lang', 'collected_at']


## 4. Explorare rapidă a corpusului
Ne uităm la canalele principale și la câteva exemple de comentarii.
Această etapă ne ajută să înțelegem ce tip de date avem înainte să folosim modelul.

In [6]:
# cele mai frecvente 15 canale sursă din dataset
df["source_channel"].value_counts().head(15).reset_index(name="n_comments")

,source_channel,n_comments
0,RecorderRomania,12177
1,turcescu111,5019
2,georgesimionoficial,3669
3,CălinGeorgescu-CanalulOficial,3460
4,@CălinGeorgescu-CanalulOficial,2557
5,TuDecizi-s3g,647
6,StareaNatiei,623
7,AltcevacuAdrianArtene,363
8,roxindaniel,305
9,otvdirect,304


In [7]:
# aruncă o privire asupra unor comentarii random din dataset
df[["source_channel", "video_title", "text"]].sample(5, random_state=42)

,source_channel,video_title,text
23002,CălinGeorgescu-CanalulOficial,Călin Georgescu - Pacea de la București ( IPJ ...,Multă sănătate dl. Președinte Călin Georgescu....
5815,@CălinGeorgescu-CanalulOficial,Călin Georgescu - De ce vorbim despre Eminescu...,Un discurs care trebuia sa vina de la Cotrocen...
11191,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,Autoritatile abilitate sa intervina!!! De acee...
11316,RecorderRomania,Primarul Negoiță a construit șosele peste magi...,In acest moment mai putem spune doar Dumnezeu ...
12505,RecorderRomania,DOCUMENTAR RECORDER. Singuri,E dureros.. e crunt.. simt vinovatie si recuno...


## 5. Alegem 10 comentarii pentru testarea promptului
Folosim 10 comentarii curate.
Puteți păstra eșantionarea aleatorie sau puteți selecta manual comentarii mai interesante.

In [8]:
sample_df = df.sample(10, random_state=44).copy()
sample_df[["source_channel", "text"]]
n_comments = 10
sample_for_prompt = sample_df.head(n_comments) 

Optional , poti alege sa folosesti  alta metoda de esantionare sau sa filtrezi dupa anumite canale sursa sau alte criterii. Important e sa ai un set de date mic pe care sa testezi promptul inainte de a-l rula pe intregul dataset.

## 6. Primul prompt exploratoriu
Completăm un prompt simplu pentru analizarea comentariilor politice.
Promptul trebuie să ceară:
- ținta comentariului;
- poziționarea față de țintă;
- tonul;
- tema;
- problema de interpretare;
- o justificare scurtă.
Important: tonul sau sentimentul general nu este același lucru cu poziționarea față de țintă.

In [9]:
# IMPORTANT - SCHIMBA PROMTUL DE MAI JOS PENTRU A SE POTRIVI CU CERINȚELE TALE ȘI ASIGURĂ-TE CĂ RESPECTĂ STRUCTURA SOLICITATĂ
# include în prompt instrucțiuni clare pentru fiecare dintre cele 7 elemente pe care vrei să le extragi și asigură-te că modelul înțelege că trebuie să returneze un JSON valid cu exact acele chei
# inlocueste "..." cu instrucțiuni clare pentru fiecare element
# Prompt de sistem: definește rolul modelului
# poti pune si alte axe de analiza care te intereseaza


SYSTEM_PROMPT = '''
Ești un asistent specializat în adnotarea comentariilor politice românești de pe YouTube.
Sarcina ta este să analizezi fiecare comentariu și să returnezi EXCLUSIV un JSON valid, fără text suplimentar și fără blocuri markdown.
Folosește exact cheile specificate.
Fii atent la ironie și sarcasm: un comentariu pozitiv ca formă poate exprima o poziție negativă față de țintă.
Distinge clar între sentimentul general al textului și poziționarea față de țintă — acestea pot fi diferite.
'''.strip()

USER_PROMPT_TEMPLATE = '''
Analizează următorul comentariu politic românesc.

Comentariu:
<<< {comment_text} >>>

Returnează EXCLUSIV un JSON valid cu exact aceste 7 chei:

- "target": entitatea politică principală vizată (persoană, partid, instituție). Dacă nu există o țintă clară, scrie "neclar". Dacă sunt mai multe ținte, alege-o pe cea principală și menționează celelalte în "interpretation_problem".
- "sentiment": polaritatea emoțională generală (valori: "pozitiv", "negativ", "mixt", "neutru").
- "stance": poziționarea față de target (valori: "favorabil", "împotrivă", "mixt", "neclar", "nu se aplică"). Atenție: sentiment negativ nu înseamnă automat stance împotrivă.
- "tone": tonul retoric dominant (ex: "ironic", "sarcastic", "admirativ", "furios", "neutru", "umoristic", "conspiraționist", "emoțional").
- "topic": tema politică scurtă (ex: "alegeri", "corupție", "economie", "politică externă").
- "interpretation_problem": sarcasm neexplicit, ținte multiple, ironie, ambiguitate — sau "niciunul".
- "reason": o propoziție care justifică etichetele alese.

Nu adăuga niciun text în afara JSON-ului.
'''.strip()

## 7. Conectarea la model
Folosim Gemini prin endpoint compatibil cu OpenAI.
Modelul și temperatura au fost setate mai sus.

In [10]:
import requests

GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={GEMINI_API_KEY}"

def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    payload = {
        "contents": [
            {
                "parts": [
                    {"text": SYSTEM_PROMPT + "\n\n" + prompt}
                ]
            }
        ],
        "generationConfig": {
            "temperature": temperature
        }
    }
    response = requests.post(
        GEMINI_URL,
        json=payload,
        headers={"Content-Type": "application/json"},
        timeout=60
    )
    response.raise_for_status()
    data = response.json()
    return data["candidates"][0]["content"]["parts"][0]["text"]

In [13]:
import requests

GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent?key={GEMINI_API_KEY}"

def annotate_comment(comment_text):
    prompt = USER_PROMPT_TEMPLATE.format(comment_text=comment_text)
    payload = {
        "contents": [
            {
                "parts": [
                    {"text": SYSTEM_PROMPT + "\n\n" + prompt}
                ]
            }
        ],
        "generationConfig": {
            "temperature": temperature
        }
    }
    response = requests.post(
        GEMINI_URL,
        json=payload,
        headers={"Content-Type": "application/json"},
        timeout=60
    )
    response.raise_for_status()
    data = response.json()
    return data["candidates"][0]["content"]["parts"][0]["text"]

## 8. Rulăm promptul pe 10 comentarii
Trimitem fiecare comentariu selectat la model și salvăm răspunsurile.

In [14]:
n_comments = 10
sample_for_prompt = sample_df.head(n_comments).copy()

outputs = []
for _, row in sample_for_prompt.iterrows():
    outputs.append({
        "source_channel": row.get("source_channel", ""),
        "video_title": row.get("video_title", ""),
        "comment_text": row["text"],
        "model_output": annotate_comment(row["text"])
    })
results_df = pd.DataFrame(outputs)
results_df

,source_channel,video_title,comment_text,model_output
0,RecorderRomania,DOCUMENTAR RECORDER. Justiție capturată,ROMANIA NU MAI EXISTA CA STAT . TOTUL ESTE DIS...,"```json\n{\n ""target"": ""statul român"",\n ""se..."
1,AltcevacuAdrianArtene,"Bebe Cotimanis, miracolul de la înmormântarea ...",Mare adevăr....în biserica atunci când preotul...,"```json\n{\n ""target"": ""neclar"",\n ""sentimen..."
2,RecorderRomania,PORTRET DE CANDIDAT: Nicușor Dan,"Nimic de Soros, nimic de globalisti, nimic de ...","```json\n{\n ""target"": ""Recorder"",\n ""sentim..."
3,RecorderRomania,EXPLICATIV RECORDER: Cazul Gânj. Cum a devenit...,interesant ca nu a vorbit nimeni despre faptul...,"```json\n{\n ""target"": ""Emil Gânj"",\n ""senti..."
4,RecorderRomania,EXPLICATIV RECORDER. Cum s-au repliat liderii ...,Deci toti vinovatii cu bani sunt liberi deși f...,"```json\n{\n ""target"": ""justiția"",\n ""sentim..."
5,RecorderRomania,Investigație Recorder: Cea mai mare firmă-fant...,"Spalare de bani, treci bani ciorditi printr-o ...","```json\n{\n ""target"": ""Registrul de Comerț"",..."
6,turcescu111,Swingeri politici în acțiune,"Au votat la fel cum au anulat alegerile: ,,fac...","```json\n{\n ""target"": ""neclar"",\n ""sentimen..."
7,turcescu111,"“O facem, dar prin batistă!”, de-asta a fost c...","Pardon, nu sunt 90 de milioane de euro,ci 900 ...","```json\n{\n ""target"": ""Orban"",\n ""sentiment..."
8,georgesimionoficial,Episodul 2: Cum ne-au furat alegerile - Turism...,Noi cunoastem adevarul ! Nu renuntam la Tara n...,"```json\n{\n ""target"": ""neclar"",\n ""sentimen..."
9,turcescu111,Swingeri politici în acțiune,"Și așa,doar de realimentare,tot nu este ok.Ace...","```json\n{\n ""target"": ""România"",\n ""sentime..."


# 9. Verificam rezultatele

In [16]:
results_df.model_output[0]

'```json\n{\n  "target": "statul român",\n  "sentiment": "negativ",\n  "stance": "împotrivă",\n  "tone": "furios",\n  "topic": "corupție",\n  "interpretation_problem": "neclar",\n  "reason": "Comentariul exprimă o frustrare profundă și o percepție de distrugere și furt la nivel național, atribuind vina celor aflați la putere și unor entități europene, sugerând o poziție fermă împotriva stării actuale a țării."\n}\n```'

In [17]:
# funcție pentru a curăța și parsa output-ul modelului, care poate conține JSON în diferite formate (text simplu sau bloc ```json)

def parse_model_output(text):
    # Modelul poate întoarce JSON ca text simplu sau în bloc ```json
    text = text.replace("```json", "")
    text = text.replace("```", "")
    text = text.strip()
    
    return json.loads(text)

In [18]:
parsed_outputs = []

for _, row in results_df.iterrows():
    parsed = parse_model_output(row["model_output"])
    
    parsed_outputs.append({
        "source_channel": row["source_channel"],
        "video_title": row["video_title"],
        "comment_text": row["comment_text"],
        "target": parsed.get("target", ""),
        "stance": parsed.get("stance", ""),
        "sentiment": parsed.get("sentiment", ""),
        "tone": parsed.get("tone", ""),
        "topic": parsed.get("topic", ""),
        "interpretation_problem": parsed.get("interpretation_problem", ""),
        "reason": parsed.get("reason", "")
    })

parsed_df = pd.DataFrame(parsed_outputs)
parsed_df

,source_channel,video_title,comment_text,target,stance,sentiment,tone,topic,interpretation_problem,reason
0,RecorderRomania,DOCUMENTAR RECORDER. Justiție capturată,ROMANIA NU MAI EXISTA CA STAT . TOTUL ESTE DIS...,statul român,împotrivă,negativ,furios,corupție,neclar,Comentariul exprimă o frustrare profundă și o ...
1,AltcevacuAdrianArtene,"Bebe Cotimanis, miracolul de la înmormântarea ...",Mare adevăr....în biserica atunci când preotul...,neclar,neclar,neutru,emoțional,religie,niciunul,Comentariul descrie o experiență personală leg...
2,RecorderRomania,PORTRET DE CANDIDAT: Nicușor Dan,"Nimic de Soros, nimic de globalisti, nimic de ...",Recorder,împotrivă,negativ,sarcastic,dezinformare,neclar,Comentariul folosește sarcasm pentru a critica...
3,RecorderRomania,EXPLICATIV RECORDER: Cazul Gânj. Cum a devenit...,interesant ca nu a vorbit nimeni despre faptul...,Emil Gânj,neclar,neutru,neutru,informații personale,niciunul,Comentariul menționează o informație despre Em...
4,RecorderRomania,EXPLICATIV RECORDER. Cum s-au repliat liderii ...,Deci toti vinovatii cu bani sunt liberi deși f...,justiția,împotrivă,negativ,furios,corupție,neclar,Comentariul exprimă frustrare și neîncredere î...
5,RecorderRomania,Investigație Recorder: Cea mai mare firmă-fant...,"Spalare de bani, treci bani ciorditi printr-o ...",Registrul de Comerț,împotrivă,negativ,furios,corupție,neclar,Comentariul acuză Registrul de Comerț de spăla...
6,turcescu111,Swingeri politici în acțiune,"Au votat la fel cum au anulat alegerile: ,,fac...",neclar,împotrivă,negativ,sarcastic,alegeri,neclar,Comentariul critică modul în care au fost vota...
7,turcescu111,"“O facem, dar prin batistă!”, de-asta a fost c...","Pardon, nu sunt 90 de milioane de euro,ci 900 ...",Orban,împotrivă,negativ,sarcastic,corupție,niciunul,Comentariul critică suma mare de bani transpor...
8,georgesimionoficial,Episodul 2: Cum ne-au furat alegerile - Turism...,Noi cunoastem adevarul ! Nu renuntam la Tara n...,neclar,neclar,pozitiv,emoțional,naționalism,niciunul,Comentariul exprimă un sentiment puternic de p...
9,turcescu111,Swingeri politici în acțiune,"Și așa,doar de realimentare,tot nu este ok.Ace...",România,împotrivă,negativ,conspiraționist,politică externă,neclar,Comentariul exprimă o poziție negativă față de...


# 10 Salvarea csv si inspectarea rezulatelor
- salvati ca csv 
- deschideti csv si verificati rezultatele 
- raspundeti la urmatoarele intrebare: promptul separă corect sentimentul general de poziționarea față de target? 

In [23]:
# salvam ca csv pentru analiza ulterioară
output_file = ROOT / "outputs" / "student_5_prompt_outputs.jsonl"
output_file.parent.mkdir(parents=True, exist_ok=True)
parsed_df.to_csv(output_file.with_suffix(".csv"), index=False)